In [1]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
sns.set(style='whitegrid',font_scale=2)
import os
import pickle
import time
os.chdir(os.getcwd())

In [2]:
# load the subsetted datasets

mb=pd.read_csv('../data/out_mb_wo_false_positives.csv.gz',low_memory=False)
#pc=pd.read_csv('../data/out_pc_preprocessed.csv',low_memory=False)
#pc_peaks=pd.read_csv('../data/ttp_pc_peaks.csv',low_memory=False)
#pc_baseline=pd.read_csv('../data/ttp_pc_baseline.csv',low_memory=False)
dm=pd.read_csv('../data/out_dm.csv.gz',low_memory=False)
vs=pd.read_csv('../data/out_vs_standardised.csv.gz',low_memory=False)
re=pd.read_csv('../data/out_re_standardised.csv.gz',low_memory=False)
lb=pd.read_csv('../data/out_lb.csv.gz',low_memory=False)
mh=pd.read_csv('../data/out_mh_standardised.csv.gz',low_memory=False)
ce=pd.read_csv('../data/out_ce_standardised_with_time.csv.gz',low_memory=False)
cm=pd.read_csv('../data/out_cm_standardised_with_drugs.csv.gz',low_memory=False) # use this dataset for the concomitant medication data
cmind=cm.copy()                          # use this dataset for the indications for the concomitant medication
ms=pd.read_csv('../data/out_ms.csv.gz',low_memory=False)
ae=pd.read_csv('../data/out_ae_standardised.csv.gz',low_memory=False)
su=pd.read_csv('../data/out_su.csv.gz',low_memory=False)
#de=pd.read_csv(dir+'//ttp_de.csv',low_memory=False,index_col=0)

## Load temporal regimens
temporal_pat_regimens=pd.read_csv('../data/out_temporal_pat_regimens_1018_20_21_22_30.csv.gz',low_memory=False)

#load patient ids
pat_ids=pd.read_csv('../data/patients_in_analysis.csv.gz',index_col=0)
pat_ids=pat_ids['USUBJID'].values.tolist()

#get drug names used in studies
#f = open(dir+'/ttp_all_drugs.pkl','rb')
#ttp_all_drugs=pickle.load(f)
#f.close()


#split up microbiolgical susceptilibilty dataset
mr=ms[ms['STD_MSTEST']=='Molecular Drug Resistance']
mic=ms[ms['STD_MSTEST']=='Minimum Inhibitory Concentration']
ms=ms[ms['STD_MSTEST']=='Microbial Susceptibility']


##### Get all the variables for all the patients

In [3]:
ttp_datasets=[mb,dm,vs,re,lb,mh,ce,cm,cmind,mr,mic,ms,ae,su]
ttp_datasets_name=['mb','dm','vs','re','lb','mh','ce','cm','cmind','mr','mic','ms','ae','su']
groupby_colnames={ 'mb':'STD_MBTEST',
                   'dm':['STUDYID','AGE','SEX','RACE','ARM'],
                   #'pc':'PCTEST',
                   'vs':'STD_VSTEST',
                   're':'STD_RETEST',
                   'lb':'LBTEST',
                   'mh':'ALL_STD_TERMS',
                   'ce':'STD_CETERM',
                   'cm':'STD_DRUGS_REPLACED',
                   'cmind':'STD_CMINDC',
                   'mr':'STD_MSAGENT',
                   'mic':'STD_MSAGENT',
                   'ms':'STD_MSAGENT',
                   'ae':'STD_AETERM',
                   'su':'STD_SUTRT'}


# 2. STEP create DATAFRAME of variables for datasets
series_l=[]
for ds,ds_name,cat in zip(ttp_datasets,groupby_colnames.keys(),groupby_colnames.values()):
    ds['USUBJID'] = ds['USUBJID'].str.replace('\\', '/', regex=False)
    if ds_name=='dm':
        l=[ds_name+'_'+x for x in groupby_colnames[ds_name]]
        series_l.append(pd.Series(l,name=ds_name))
    else:
        l=[ds_name+'_'+x for x in ds[cat].value_counts().index.tolist()]
        series_l.append(pd.Series(l,name=ds_name))

variables_df=pd.concat(series_l,axis=1)
variables_df.to_excel('../data/variables_df.xlsx')

all_variables=[]
for col in variables_df:
    l=variables_df[col].dropna()
    all_variables=all_variables+l.tolist()

## Collect the names of all variables available for each patient along with dataframes containing the data for the variables 

In [4]:
len(ttp_datasets)

14

In [4]:
all_pat_variables_dict={}
start = time.time()

for ds,ds_name in zip(ttp_datasets[0:],ttp_datasets_name[0:]):
    all_pat_variables_dict[ds_name]={}
    common_ids=list(set(pat_ids)&set(ds['USUBJID'].values))
    print(ds_name)
    if ds_name=='dm':
        for pat_id in common_ids:
            bulk_df=ds[ds['USUBJID']==pat_id]       
            dm_vars=bulk_df.loc[:,groupby_colnames[ds_name]].dropna(how='all',axis=1).columns.tolist()
            all_pat_variables_dict[ds_name][pat_id]=dm_vars
    
    if ds_name!='dm':
        groupby_cat=groupby_colnames[ds_name]
        for pat_id in common_ids:
            bulk_df=ds[ds['USUBJID']==pat_id]
            
            ## Lab variables were split to numerical and categorical standardised variables-> concatenate them
            if ds_name=='lb':
                variables_numerical=bulk_df['STD_NUM_TEST'].value_counts().index.tolist()
                variables_categorical=bulk_df['STD_CAT_TEST'].value_counts().index.tolist()
                variables=list(set(variables_numerical+variables_numerical))
            if ds_name!='lb':
                variables=bulk_df[groupby_cat].value_counts().index.tolist()
            all_pat_variables_dict[ds_name][pat_id]=[ds_name+'_'+x for x in variables]
end = time.time()
print((end - start)/60),"minutes runtime"

f = open('../data/all_pat_variables_dict',"wb")
pickle.dump(all_pat_variables_dict,f)
f.close()

mb
dm
vs
re
lb
mh
ce
cm
cmind
mr
mic
ms
ae
su
3.925475748380025


In [16]:
all_pat_variables_dict['mb']['TB-1021/1035711']#.keys()

['mb_ZN-smear', 'mb_MGIT', 'mb_LJ-culture', 'mb_AccuProbe']

#### create a dataframe with patients IDs as rows, and all the variables available in the datasets in the columns. If a variable has values for the patient, indicate it with a 1, 0 if variable doesn’t have a value for the patient.

In [5]:
from tqdm import tqdm

f = open('../data/all_pat_variables_dict',"rb")
all_pat_variables_dict=pickle.load(f)


all_variables=[]
for col in variables_df:
    l=variables_df[col].dropna()
    all_variables=all_variables+l.tolist()

variables_per_patient_all=pd.DataFrame(index=pat_ids,columns=all_variables)

start = time.time()
num=0

variables_per_patient_all=variables_per_patient_all.dropna(how='all',axis=1)
variables_per_patient_all=variables_per_patient_all.fillna(int(0))
variables_per_patient_all


for id in tqdm(pat_ids[:]):

    l=[]
    for ds_name in ttp_datasets_name[:]:
        ds_dict=all_pat_variables_dict[ds_name]    

        try:
            vars=ds_dict[id]
            l.extend(vars)
        except KeyError:
            continue
    
    variables_per_patient_all.loc[id,l]=int(1)
    
end = time.time()
print((end - start)/60,'minutes runtime')


100%|██████████| 5796/5796 [11:19<00:00,  8.54it/s]

11.325258421897889 minutes runtime


In [6]:
variables_per_patient_all.to_csv('../data/all_pat_variables.csv.gz',compression='gzip')

In [20]:
all_pat_variables_dict['mb']['TB-1021/1035711']#.keys()

['mb_ZN-smear', 'mb_MGIT', 'mb_LJ-culture', 'mb_AccuProbe']

In [13]:
variables_per_patient_all

,mb_MGIT,mb_ZN-smear,mb_HAIN-test,mb_MTB-complex,mb_Auramine-smear,mb_LJ-culture,mb_AccuProbe,mb_MPT64-Antigen-Test,mb_RT-PCR,STUDYID,...,ae_CREPITATIONS LEFT LOWER ZONE,ae_NECK RASH,ae_RIGHT FOREFINGER RINGWORM,ae_DIFFUSE MACULA PAPULAR FACIAL RASH,ae_PAINFUL TONGUE,ae_SENSORY PERIPHERAL NERUOPATHY,ae_FLUSHING OF SKIN GENERALLY,ae_ETHAMBUTOL ASSOCIATED TOXIC OPTIC NEUROPATHY,ae_CONJUNCTIVAL HAEMORRHAGE RIGHT EYE,su_SMOKING EVER
TB-1020/1016,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
TB-1020/1023,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
TB-1020/1029,NaN,1.0,NaN,1.0,NaN,NaN,NaN,NaN,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
TB-1020/1031,1.0,1.0,NaN,1.0,NaN,NaN,NaN,NaN,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
TB-1020/1039,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TB-1018/04-9011,1.0,NaN,NaN,1.0,1.0,NaN,NaN,1.0,1.0,1.0,...,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
TB-1018/04-9010,1.0,NaN,NaN,NaN,1.0,NaN,NaN,1.0,1.0,1.0,...,NaN,NaN,NaN,1.0,1.0,NaN,NaN,NaN,NaN,NaN
TB-1018/02-9066,1.0,NaN,1.0,NaN,1.0,NaN,NaN,1.0,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1.0,NaN,NaN
TB-1018/02-9046,1.0,NaN,1.0,NaN,1.0,NaN,NaN,1.0,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN
